Feature engineering for the behaviour-cloning networks.
Used by BOTH training (dataset building) and the policy at run time,
so features are always computed identically.

In [24]:
import numpy as np
%load_ext autoreload
%autoreload 2
import sys; sys.path.insert(0, "/home/yousef/ucl_ws/comp0245-final/robotnik_arm_task/yousef")
from features import *
from dataset import split_episodes
from dataset import build_dataset
from models import MLP
from train import make_loaders, train_model
import torch

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
print (world_to_base(np.array([[0., 1.]]), np.array([np.pi/2])))

[[1.000000e+00 6.123234e-17]]


In [13]:
d = np.load("/home/yousef/ucl_ws/comp0245-final/robotnik_arm_task/data/raw/ep_00000.npz", 'r')
Xb, nb = base_features(d)
print(Xb[d["phase"] == 3, 2].max()) 

0.07061196


In [14]:
w = d["wheel_cmds"]
print(np.abs(vw_to_wheels(wheels_to_vw(w)) - w).max())

2.3841858e-07


In [15]:
Xa, na = arm_features(d)
print (Xb.shape, Xa.shape, len(d["t"]), np.isnan(Xb).any(), np.isnan(Xa).any())

(14061, 10) (14061, 25) 14061 False False


In [16]:
import glob
worst = 0
for f in sorted(glob.glob(".../data/raw/ep_0*.npz"))[:100]:
    e = np.load(f)
    if e["success"]:
        Xb_, _ = base_features(e)
        worst = max(worst, Xb_[e["phase"] == 3, 2].max())
print(worst)        # should still be ≲ 0.08 (a bit more is OK for a few timeout episodes)

0


In [17]:
files = sorted(glob.glob("/home/yousef/ucl_ws/comp0245-final/robotnik_arm_task/data/raw/ep_0*.npz"))[:100]
print(len(files))          # should be 100

worst = 0
for f in files:
    e = np.load(f)
    if e["success"]:
        Xb_, _ = base_features(e)
        worst = max(worst, Xb_[e["phase"] == 3, 2].max())
print(worst)

100
0.16069253


In [18]:
for f in files:
    e = np.load(f)
    if e["success"]:
        Xb_, _ = base_features(e)
        m = Xb_[e["phase"] == 3, 2].max()
        if m > 0.08:
            print(f[-12:], f"max dist={m:.3f}", f"nav_time={float(e['nav_time']):.1f}s")

ep_00009.npz max dist=0.080 nav_time=8.6s
ep_00013.npz max dist=0.091 nav_time=5.1s
ep_00015.npz max dist=0.107 nav_time=4.6s
ep_00016.npz max dist=0.081 nav_time=8.7s
ep_00017.npz max dist=0.091 nav_time=8.3s
ep_00018.npz max dist=0.080 nav_time=7.3s
ep_00019.npz max dist=0.081 nav_time=7.0s
ep_00020.npz max dist=0.085 nav_time=6.6s
ep_00022.npz max dist=0.089 nav_time=6.6s
ep_00023.npz max dist=0.087 nav_time=4.0s
ep_00025.npz max dist=0.110 nav_time=4.0s
ep_00029.npz max dist=0.080 nav_time=5.0s
ep_00030.npz max dist=0.082 nav_time=6.5s
ep_00031.npz max dist=0.110 nav_time=6.9s
ep_00033.npz max dist=0.158 nav_time=6.3s
ep_00034.npz max dist=0.080 nav_time=2.7s
ep_00039.npz max dist=0.080 nav_time=11.3s
ep_00040.npz max dist=0.081 nav_time=1.8s
ep_00050.npz max dist=0.103 nav_time=7.5s
ep_00054.npz max dist=0.161 nav_time=6.6s
ep_00056.npz max dist=0.138 nav_time=6.2s
ep_00057.npz max dist=0.116 nav_time=8.2s
ep_00059.npz max dist=0.093 nav_time=5.6s
ep_00061.npz max dist=0.080 nav_t

In [19]:
train_f, val_f, test_f = split_episodes()
print(len(train_f), len(val_f), len(test_f))

1008 177 98


In [20]:
print(set(train_f) & set(val_f), set(train_f) & set(test_f))   # both should be set()

set() set()


In [21]:
D = build_dataset(step=10)
for k, v in D.items():
    print(f"{k:12s} {v.shape}")
print(any(np.isnan(v).any() for v in D.values()))   # should be False


saved /home/yousef/ucl_ws/comp0245-final/robotnik_arm_task/data/processed_step10.npz
Xb_train     (1026259, 10)
Yb_train     (1026259, 2)
Xa_train     (1026259, 25)
Ya_train     (1026259, 7)
phase_train  (1026259,)
ep_id_train  (1026259,)
Xb_val       (175714, 10)
Yb_val       (175714, 2)
Xa_val       (175714, 25)
Ya_val       (175714, 7)
phase_val    (175714,)
ep_id_val    (175714,)
Xb_test      (99966, 10)
Yb_test      (99966, 2)
Xa_test      (99966, 25)
Ya_test      (99966, 7)
phase_test   (99966,)
ep_id_test   (99966,)
Xb_mu        (10,)
Xb_sd        (10,)
Yb_mu        (2,)
Yb_sd        (2,)
Xa_mu        (25,)
Xa_sd        (25,)
Ya_mu        (7,)
Ya_sd        (7,)
False


In [27]:
m = MLP(10, 2); print(m)
print(m(torch.zeros(5, 10)).shape)     # → torch.Size([5, 2])

MLP(
  (net): Sequential(
    (0): Linear(in_features=10, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=256, bias=True)
    (3): ReLU()
    (4): Linear(in_features=256, out_features=256, bias=True)
    (5): ReLU()
    (6): Linear(in_features=256, out_features=2, bias=True)
  )
)
torch.Size([5, 2])


In [28]:
train_dl, val_dl = make_loaders(D, "b")
base_net, base_hist = train_model(MLP(10, 2), train_dl, val_dl)

epoch  1 train 0.0176 val0.0059
epoch  2 train 0.0040 val0.0037
epoch  3 train 0.0028 val0.0029
epoch  4 train 0.0023 val0.0031
epoch  5 train 0.0021 val0.0024
epoch  6 train 0.0020 val0.0036
epoch  7 train 0.0018 val0.0021
epoch  8 train 0.0017 val0.0022
epoch  9 train 0.0016 val0.0026
epoch 10 train 0.0014 val0.0022
epoch 11 train 0.0014 val0.0020
epoch 12 train 0.0014 val0.0023
epoch 13 train 0.0013 val0.0029
epoch 14 train 0.0013 val0.0021
epoch 15 train 0.0012 val0.0018
epoch 16 train 0.0012 val0.0017
epoch 17 train 0.0011 val0.0019
epoch 18 train 0.0012 val0.0019
epoch 19 train 0.0011 val0.0020
epoch 20 train 0.0011 val0.0016
epoch 21 train 0.0011 val0.0018
epoch 22 train 0.0010 val0.0027
epoch 23 train 0.0011 val0.0017
epoch 24 train 0.0010 val0.0020
epoch 25 train 0.0010 val0.0021
early stop
